# Encoder #2 — causal two-sample Siamese on raw sequences

The orthogonal-input bet. MLP-on-features was redundant (corr 0.87, [[mlp-probe-features-dead]]); a net can only add value if it sees something the 1072 aggregate features flatten away — **raw sequence shape**.

**Design**
- Reference (period-1 hist) encoded once -> vector `r`. Online (period-2) streamed through a **causal GRU** -> hidden `h_t` = summary of `online[:t]`.
- **Normalize both by PRE (reference) stats only** — variance/scale/AR shift survives into the net (the Chronos lesson, [[chronos-foundation-model-dead-end]]).
- Per-step head on `[h_t, r, h_t-r, h_t*r]` -> logit_t. Label is **per-step** (turns on at `tau_index`).
- Streaming-native: GRU carries state O(1)/step, matches the submission constraint.

**Gate:** holdout TS-AUC vs the reproducible bar **0.579** ([[structural-break-baseline-0579-not-0606]]), rank-corr vs GBT, 50/50 blend. Low corr earns a blend slot even if it loses solo.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys, time, json
import numpy as np, pandas as pd
import torch, torch.nn as nn

HERE = os.path.abspath("")
EXPERIMENTS = os.path.abspath(os.path.join(HERE, "..", ".."))
sys.path.insert(0, EXPERIMENTS)
import sb_common as C

DATA = EXPERIMENTS.rsplit("/experiments", 1)[0]
CACHE = os.path.join(HERE, "seq_cache.npz")
L_REF = 512
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
SEED = 0
print("device", DEVICE, "| data", DATA)

device mps | data /Users/minqi/Documents/ADIA_Lab_Structural_Break_Challenge


## Build raw-sequence cache (once)

Per series: downsampled reference channels `[z, z^2, |z|]` (L=512), online per-step features `[z, z^2, |z|, dz, run_mean, run_var]` (all causal, normalized by ref stats), ref scalars, per-step target, times. Cached to `seq_cache.npz`.

In [2]:
EPS = 1e-9
def build_cache():
    X = pd.read_parquet(f"{DATA}/X_train.parquet")
    y = pd.read_parquet(f"{DATA}/y_train.parquet")["target"]   # ONLINE rows only, sorted (id,time)
    ids_all = X.index.get_level_values("id").to_numpy()
    times_all = X.index.get_level_values("time").to_numpy()
    vals = X["value"].to_numpy(np.float64)
    per = X["period"].to_numpy()
    yv = y.to_numpy().astype(np.int8)          # aligned to X online rows in file order
    uids, starts = np.unique(ids_all, return_index=True)
    bounds = np.append(starts, len(ids_all))
    out = {"id": [], "ref": [], "refscalar": [], "onl": [], "tgt": [], "time": [], "len": []}
    oc = 0                                      # running cursor into the online-only y
    for k, sid in enumerate(uids):
        s, e = bounds[k], bounds[k + 1]
        m_on = per[s:e] == 2
        n_on = int(m_on.sum())
        tgt_full = yv[oc:oc + n_on]; oc += n_on   # advance for EVERY series (even skipped)
        ref = vals[s:e][~m_on]
        x = vals[s:e][m_on]
        t = times_all[s:e][m_on]
        if len(x) == 0 or len(ref) < 8:
            continue
        mean, std = ref.mean(), ref.std() + EPS
        di = np.linspace(0, len(ref) - 1, L_REF).astype(np.int64)
        zr = (ref[di] - mean) / std
        refch = np.stack([zr, zr * zr, np.abs(zr)]).astype(np.float32)   # [3, L_REF]
        ar1 = np.corrcoef(ref[:-1], ref[1:])[0, 1] if len(ref) > 2 else 0.0
        kurt = float(pd.Series(ref).kurt()) if len(ref) > 3 else 0.0
        refsc = np.array([mean, std, np.nan_to_num(ar1), np.nan_to_num(kurt),
                          np.log(len(ref))], np.float32)
        z = (x - mean) / std
        n = np.arange(1, len(z) + 1)
        rm = np.cumsum(z) / n
        rv = np.clip(np.cumsum(z * z) / n - rm * rm, 0, None)
        dz = np.concatenate([[0.0], np.diff(z)])
        onl = np.stack([z, z * z, np.abs(z), dz, rm, rv], 1).astype(np.float32)  # [T,6]
        out["id"].append(int(sid)); out["ref"].append(refch); out["refscalar"].append(refsc)
        out["onl"].append(onl); out["tgt"].append(tgt_full.astype(np.int8))
        out["time"].append(t.astype(np.int64)); out["len"].append(len(z))
    np.savez(CACHE,
             ids=np.array(out["id"]), ref=np.stack(out["ref"]),
             refscalar=np.stack(out["refscalar"]),
             onl=np.array(out["onl"], dtype=object), tgt=np.array(out["tgt"], dtype=object),
             time=np.array(out["time"], dtype=object), lens=np.array(out["len"]))
    print("cached", len(out["id"]), "series | online rows consumed", oc, "of", len(yv))

if not os.path.exists(CACHE):
    t0 = time.time(); build_cache(); print(f"build {time.time()-t0:.0f}s")
else:
    print("cache exists")

cache exists


In [3]:
Z = np.load(CACHE, allow_pickle=True)
ids = Z["ids"]; ref = Z["ref"].astype(np.float32); refsc = Z["refscalar"].astype(np.float32)
onl = Z["onl"]; tgt = Z["tgt"]; tim = Z["time"]; lens = Z["lens"]
id2i = {int(s): i for i, s in enumerate(ids)}

tr_ids = set(np.load(f"{EXPERIMENTS}/tr_ids.npy", allow_pickle=True).tolist())
tr_mask = np.array([int(s) in tr_ids for s in ids])

# ROBUST per-channel scaling (median / IQR) on TRAIN, then clip. Channels z^2, run_var,
# z have extreme tails (degenerate near-zero-std references blow up normalization); plain
# mean/std standardization squashed ~100% of z^2 / run_var to 0 and killed all signal.
def robust(x, axis):
    med = np.median(x, axis=axis, keepdims=True)
    q1, q3 = np.percentile(x, 25, axis=axis, keepdims=True), np.percentile(x, 75, axis=axis, keepdims=True)
    return med, (q3 - q1) + 1e-6

onl_cat = np.concatenate([onl[i] for i in np.where(tr_mask)[0]])
_m, _s = robust(onl_cat, 0); onl_mu, onl_sd = _m[0], _s[0]           # [6]
_rm, _rs = robust(ref[tr_mask], (0, 2)); ref_mu, ref_sd = _rm, _rs   # [1,3,1]
_sm, _ss = robust(refsc[tr_mask], 0); sc_mu, sc_sd = _sm[0], _ss[0]  # [5]

ref_n = np.clip((ref - ref_mu) / ref_sd, -5, 5).astype(np.float32)
refsc_n = np.clip((refsc - sc_mu) / sc_sd, -5, 5).astype(np.float32)
print("series", len(ids), "| train", tr_mask.sum(), "| onl feat dim", onl_mu.shape[0])
print("onl IQR scale:", np.round(onl_sd, 3))

series 10000 | train 8000 | onl feat dim 6
onl IQR scale: [1.271 1.125 0.804 1.642 0.094 0.278]


## Model — ref conv encoder + online **causal TCN** + compare head

Online branch is a dilated **causal** TCN (not an RNN): parallel over time, no `pack_padded` (which is pathologically slow/hangs on MPS). Causal = left-pad only, so `h_t` depends solely on `online[:t]`. Input already carries causal running mean/var, so a modest receptive field suffices.

In [4]:
class RefEncoder(nn.Module):
    def __init__(self, n_scalar=5, e=48):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(3, 24, 5, padding=2), nn.ReLU(), nn.Conv1d(24, 24, 5, padding=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        self.head = nn.Sequential(nn.Linear(24 + n_scalar, e), nn.ReLU())
    def forward(self, refch, refsc):
        c = self.conv(refch).squeeze(-1)
        return self.head(torch.cat([c, refsc], 1))

class CausalConv1d(nn.Module):
    def __init__(self, cin, cout, k=3, d=1):
        super().__init__()
        self.pad = (k - 1) * d
        self.conv = nn.Conv1d(cin, cout, k, dilation=d)
    def forward(self, x):                      # x [B,C,T]
        return self.conv(nn.functional.pad(x, (self.pad, 0)))

class OnlineTCN(nn.Module):
    def __init__(self, cin=6, C=64, dils=(1, 2, 4, 8, 16, 32), p=0.1):
        super().__init__()
        layers, c = [], cin
        for d in dils:
            layers += [CausalConv1d(c, C, 3, d), nn.ReLU(), nn.Dropout(p)]; c = C
        self.net = nn.Sequential(*layers)
    def forward(self, x):                      # x [B,T,cin] -> [B,T,C]
        return self.net(x.transpose(1, 2)).transpose(1, 2)

class Siamese(nn.Module):
    def __init__(self, d_onl=6, e=48, C=64, p=0.4):
        super().__init__()
        self.refenc = RefEncoder(e=e)
        self.tcn = OnlineTCN(d_onl, C)
        self.rproj = nn.Linear(e, C)
        self.head = nn.Sequential(nn.Linear(4 * C, 96), nn.ReLU(), nn.Dropout(p), nn.Linear(96, 1))
    def forward(self, refch, refsc, onl, gidx=None):
        r = self.rproj(self.refenc(refch, refsc))      # [B, C]
        h = self.tcn(onl)                              # [B, T, C]  (TCN is cheap; head is not)
        if gidx is not None:                           # gather grid steps -> run head only there
            h = torch.gather(h, 1, gidx[..., None].expand(-1, -1, h.shape[-1]))  # [B, G, C]
        rb = r[:, None, :].expand(-1, h.shape[1], -1)
        feat = torch.cat([h, rb, h - rb, h * rb], -1)
        return self.head(feat).squeeze(-1)             # [B, G] (train) or [B, T] (eval)

In [5]:
import sb_bank as B
STEP_GRID = B.default_step_grid()                      # log-spaced online steps (same as GBT)
# per-series grid step positions (subset of 0..len-1) + their targets
grid_pos = [STEP_GRID[STEP_GRID < int(lens[i])].astype(np.int64) for i in range(len(ids))]
grid_tgt = [tgt[i][grid_pos[i]].astype(np.float32) for i in range(len(ids))]

def pad_onl(bi):
    L = [int(lens[i]) for i in bi]; mx = max(L); Bn = len(bi)
    onl_b = np.zeros((Bn, mx, onl_mu.shape[0]), np.float32)
    for j, i in enumerate(bi):
        onl_b[j, :L[j]] = np.clip((onl[i] - onl_mu) / onl_sd, -5, 5)
    return torch.from_numpy(ref_n[bi]), torch.from_numpy(refsc_n[bi]), torch.from_numpy(onl_b), L

def collate_eval(bi):                                  # full-T (for holdout scoring)
    refb, rscb, onlb, L = pad_onl(bi)
    return refb, rscb, onlb, L

def collate_train(bi):                                 # gather grid steps -> head runs on ~22 steps
    refb, rscb, onlb, L = pad_onl(bi)
    Bn = len(bi); G = max(len(grid_pos[i]) for i in bi)
    gidx = np.zeros((Bn, G), np.int64); gt = np.zeros((Bn, G), np.float32)
    gm = np.zeros((Bn, G), np.float32); gw = np.zeros((Bn, G), np.float32)
    for j, i in enumerate(bi):
        g = grid_pos[i]; ng = len(g)
        gidx[j, :ng] = g; gt[j, :ng] = grid_tgt[i]; gm[j, :ng] = 1.0; gw[j, :ng] = 1.0 / ng
    return (refb, rscb, onlb, torch.from_numpy(gidx),
            torch.from_numpy(gt), torch.from_numpy(gm), torch.from_numpy(gw))

# holdout labels/rows identical to the probe (feat_final_holdout)
y_full = C.load_y()
Fh_idx = pd.read_parquet(f"{EXPERIMENTS}/feat_final_holdout.parquet", columns=[]).index
yh = pd.Series(y_full.reindex(Fh_idx).to_numpy().astype(np.int8), index=Fh_idx)
va_idx = np.where(~tr_mask)[0]
tr_idx_arr = np.where(tr_mask)[0]

# map each holdout row -> (series cache-index ci, local online position k), aligned to Fh_idx.
hid = Fh_idx.get_level_values("id").to_numpy(); htime = Fh_idx.get_level_values("time").to_numpy()
row_ci = np.empty(len(Fh_idx), np.int64); row_k = np.empty(len(Fh_idx), np.int64)
_tpos = {}
for r in range(len(Fh_idx)):
    ci = id2i[int(hid[r])]
    d = _tpos.get(ci)
    if d is None:
        d = {int(t): p for p, t in enumerate(tim[ci])}; _tpos[ci] = d
    row_ci[r] = ci; row_k[r] = d[int(htime[r])]
print("holdout rows", len(yh), "| va series", len(va_idx), "| grid steps/series ~", len(grid_pos[va_idx[0]]))

holdout rows 43679 | va series 2000 | grid steps/series ~ 26


In [6]:
@torch.no_grad()
def predict_holdout(model, bs=256):
    """Score only the holdout rows. Run each va series once (length-bucketed, full-T),
    cache per-step probs, gather mapped (ci,k) -> Series aligned to Fh_idx."""
    model.eval()
    order = va_idx[np.argsort(lens[va_idx])]        # bucket by length -> minimal pad
    probs = {}
    for i in range(0, len(order), bs):
        bidx = order[i:i+bs]
        refb, rscb, onlb, L = collate_eval(bidx)
        p = torch.sigmoid(model(refb.to(DEVICE), rscb.to(DEVICE), onlb.to(DEVICE))).cpu().numpy()
        for j, ci in enumerate(bidx):
            probs[ci] = p[j, :int(lens[ci])]
    res = np.array([probs[row_ci[r]][row_k[r]] for r in range(len(Fh_idx))], np.float32)
    return pd.Series(res, index=Fh_idx)

def evaluate(model):
    pred = predict_holdout(model)
    return C.ts_auc(pred, yh), pred

## Train

In [7]:
def make_buckets(idx, bs):
    """Sort by length, cut into bs-sized buckets -> homogeneous lengths, minimal pad."""
    order = idx[np.argsort(lens[idx])]
    return [order[i:i+bs] for i in range(0, len(order), bs)]

def rank_loss(logit, gt, gm):
    """Per-step pairwise soft-AUC. Column g == online step STEP_GRID[g] for every series,
    so each column is a TS-AUC stratum. Contrast pos vs neg within each column."""
    d = logit[:, None, :] - logit[None, :, :]                      # [B,B,G]
    pv = (gm[:, None, :] * gt[:, None, :]) * (gm[None, :, :] * (1 - gt[None, :, :]))
    return -(torch.nn.functional.logsigmoid(d) * pv).sum() / pv.sum().clamp_min(1.0)

def train_one(seed, epochs=12, bs=128, lr=2e-3, wd=3e-4, eval_every=2):
    torch.manual_seed(seed); np.random.seed(seed)
    model = Siamese().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    buckets = make_buckets(tr_idx_arr, bs); steps = len(buckets)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs, steps_per_epoch=steps)
    best = (0.0, -1); best_pred = None; t0 = time.time()
    for ep in range(epochs):
        model.train(); tot = 0.0
        for b in np.random.permutation(len(buckets)):
            bidx = buckets[b]
            refb, rscb, onlb, gidx, gt, gm, gw = collate_train(bidx)
            refb, rscb, onlb, gidx = refb.to(DEVICE), rscb.to(DEVICE), onlb.to(DEVICE), gidx.to(DEVICE)
            gt, gm = gt.to(DEVICE), gm.to(DEVICE)
            opt.zero_grad()
            loss = rank_loss(model(refb, rscb, onlb, gidx), gt, gm)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sched.step(); tot += loss.item()
        if ep % eval_every == 0 or ep == epochs - 1:
            auc, pred = evaluate(model)
            if auc > best[0]:
                best = (auc, ep); best_pred = pred
            print(f"  seed{seed} ep{ep:02d}  loss {tot/steps:.4f}  holdout {auc:.4f}  (best {best[0]:.4f}@{best[1]})  {time.time()-t0:.0f}s", flush=True)
    return best, best_pred

# 3-seed bag (decorrelated weak learners average well)
seed_preds, seed_aucs = [], []
for s in (0, 1, 2):
    (a, e), p = train_one(s)
    seed_aucs.append(a); seed_preds.append(p)
    print(f"seed{s} best {a:.4f} (ep{e})", flush=True)
enc_pred = pd.Series(np.mean([C.rank01(p.reindex(Fh_idx).to_numpy()) for p in seed_preds], axis=0), index=Fh_idx)
best_auc = C.ts_auc(enc_pred, yh); best_ep = -1
print(f"per-seed {np.round(seed_aucs,4)} | 3-seed BAG {best_auc:.4f}", flush=True)

  seed0 ep00  loss 0.6929  holdout 0.5221  (best 0.5221@0)  13s


  seed0 ep02  loss 0.6904  holdout 0.5311  (best 0.5311@2)  28s


  seed0 ep04  loss 0.6903  holdout 0.5004  (best 0.5311@2)  37s


  seed0 ep06  loss 0.6868  holdout 0.5312  (best 0.5312@6)  50s


  seed0 ep08  loss 0.6829  holdout 0.5304  (best 0.5312@6)  64s


...

  seed2 ep08  loss 0.6826  holdout 0.5288  (best 0.5305@4)  68s


  seed2 ep10  loss 0.6793  holdout 0.5274  (best 0.5305@4)  81s


  seed2 ep11  loss 0.6789  holdout 0.5274  (best 0.5305@4)  88s


seed2 best 0.5305 (ep4)


per-seed [0.5312 0.5309 0.5305] | 3-seed BAG 0.5313


## Verdict — vs GBT 0.579, decorrelation, blend

In [8]:
# GBT baseline on same holdout rows (fresh 3-seed, same as probe)
import lightgbm as lgb
PARAMS = json.load(open(f"{EXPERIMENTS}/models/bank_lgbm/params.json"))
F = pd.read_parquet(f"{EXPERIMENTS}/feat_final_train.parquet"); cols = list(F.columns)
fids = F.index.get_level_values("id").to_numpy(); mtr = np.isin(fids, list(tr_ids))
Xtr = F.to_numpy(np.float32)[mtr]; ytr = y_full.reindex(F.index).to_numpy().astype(np.int8)[mtr]
wtr = C.eq_series_weight(F.index[mtr]).astype(np.float32); del F
Fh = pd.read_parquet(f"{EXPERIMENTS}/feat_final_holdout.parquet")[cols]
Xh = Fh.to_numpy(np.float32); h_idx = Fh.index; del Fh
gp = [C.rank01(lgb.LGBMClassifier(random_state=s, **PARAMS).fit(Xtr, ytr, sample_weight=wtr)
               .predict_proba(Xh)[:, 1]) for s in (0, 1, 2)]
gbt_pred = pd.Series(np.mean(gp, axis=0), index=h_idx)
gbt_auc = C.ts_auc(gbt_pred, yh)
print(f"GBT holdout {gbt_auc:.4f}")

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


GBT holdout 0.5793


In [9]:
enc_h = enc_pred.reindex(h_idx)
corr = np.corrcoef(C.rank01(enc_h.to_numpy()), gbt_pred.to_numpy())[0, 1]
blend = pd.Series(0.5*C.rank01(enc_h.to_numpy()) + 0.5*gbt_pred.to_numpy(), index=h_idx)
blend_auc = C.ts_auc(blend, yh)
print(f"Encoder holdout TS-AUC {best_auc:.4f} (epoch {best_ep})")
print(f"GBT     holdout TS-AUC {gbt_auc:.4f}")
print(f"gap     {best_auc-gbt_auc:+.4f}   rank-corr {corr:.3f}")
print(f"50/50 blend          {blend_auc:.4f}  ({blend_auc-gbt_auc:+.4f} vs GBT)")

enc_pred.to_frame("enc").to_parquet(os.path.join(HERE, "encoder_holdout_pred.parquet"))
json.dump({"enc_auc": float(best_auc), "enc_epoch": int(best_ep), "gbt_auc": float(gbt_auc),
           "corr": float(corr), "blend_auc": float(blend_auc)},
          open(os.path.join(HERE, "result.json"), "w"), indent=2)

Encoder holdout TS-AUC 0.5313 (epoch -1)
GBT     holdout TS-AUC 0.5793
gap     -0.0480   rank-corr 0.351
50/50 blend          0.5489  (-0.0304 vs GBT)
